# WBG Chatbot Bias Evaluation

**Instructions:**
1. Go to `File → Save a copy in Drive` to get your own copy
2. Run Cell 1 — clones the repo and mounts your Drive
3. Follow the cells in order

In [ ]:
# CELL 1 — Setup & imports
# ============================================================
# Clones the exercise repo into /content/ so all source files
# and input configs are immediately available.
# Outputs are saved to the user's own Google Drive.
# ============================================================

%load_ext autoreload
%autoreload 2

import sys
import os
import json
from collections import defaultdict

# ── Clone repo ───────────────────────────────────────────────
REPO_URL = 'https://github.com/YOUR_ORG/YOUR_REPO.git'
REPO_DIR = '/content/wbg-bias-exercise'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ Repository cloned.')
else:
    !git -C {REPO_DIR} pull
    print('✅ Repository updated.')

SRC_DIR   = f'{REPO_DIR}/src'
INPUT_DIR = f'{REPO_DIR}/input'
sys.path.insert(0, SRC_DIR)

# ── Mount Drive for personal outputs ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/wbg_bias_exercise/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Local imports ─────────────────────────────────────────────
from models             import Prompt, Response, TemplateResult
from config_loader      import ConfigLoader
from prompt_expander    import PromptExpander
from prompt_exporter    import PromptExporter
from response_loader    import ResponseLoader
from signal_classifier  import SignalClassifier
from llm_judge          import LLMJudge
from cascade_classifier import CascadeClassifier
from bias_analyser      import BiasAnalyser
from results_printer    import ResultsPrinter

# ── Path constants ────────────────────────────────────────────
TEMPLATES_PATH    = f'{INPUT_DIR}/templates.json'
COMMUNITIES_PATH  = f'{INPUT_DIR}/communities.json'
RESPONSES_PATH    = f'{OUTPUT_DIR}/response_collection_template.json'
CSV_RESULTS_PATH  = f'{OUTPUT_DIR}/bias_evaluation_results.csv'
PLOT_PATH         = f'{OUTPUT_DIR}/plot_results.png'

JUDGE_API_URL = 'http://PLACEHOLDER_ADDRESS/execute_prompt'
JUDGE_MODEL   = 'gpt-4o'

print('\n✅ Setup complete.')
print(f'   Repo    : {REPO_DIR}')
print(f'   Outputs : {OUTPUT_DIR}')

In [ ]:
# CELL 2 — Load configuration
# ============================================================
# Reads templates and communities from the repo input folder.
# ============================================================

loader = ConfigLoader(
    templates_path   = TEMPLATES_PATH,
    communities_path = COMMUNITIES_PATH,
)

TEMPLATES, COMMUNITIES = loader.load()
ConfigLoader.summarise(TEMPLATES, COMMUNITIES)

In [ ]:
# CELL 3 — Generate and preview prompts
# ============================================================
# Expands every template across all community value
# combinations and prints a structured preview.
# ============================================================

expander    = PromptExpander(TEMPLATES, COMMUNITIES)
ALL_PROMPTS = expander.expand_all()

by_template = defaultdict(list)
for p in ALL_PROMPTS:
    by_template[p.template_id].append(p)

print('=' * 68)
print(f'  PROMPTS GENERATED — {len(ALL_PROMPTS)} total')
print('=' * 68)
for tid, prompts in sorted(by_template.items()):
    print(f'\n  ▸ {tid}  ({len(prompts)} prompts)')
    print(f'    Reference: {prompts[0].reference_template}\n')
    for p in prompts:
        vals_str = '  |  '.join(f'{k} = {v}' for k, v in p.values.items())
        print(f'    [{p.prompt_id}]  {vals_str}')
        print(f'           {p.prompt}\n')

In [ ]:
# CELL 4 — Export prompts for manual testing
# ============================================================
# Writes two files to the output folder:
#   prompts_for_testing.csv           — print-friendly table
#   response_collection_template.json — fill in after testing
# The CSV is downloaded automatically in Colab.
# Existing template groups with filled responses are preserved.
# ============================================================

PromptExporter(ALL_PROMPTS).export_all(
    csv_path  = CSV_RESULTS_PATH.replace('bias_evaluation_results', 'prompts_for_testing'),
    json_path = RESPONSES_PATH,
)

In [ ]:
# CELL 5 — Load responses
# ============================================================
# Reads the completed response JSON from the output folder.
# Finds the file on disk automatically if it exists in Drive,
# otherwise opens the Colab upload dialog.
# ============================================================

loader    = ResponseLoader(path=RESPONSES_PATH)
RESPONSES = loader.load()
ResponseLoader.summarise(RESPONSES)

In [ ]:
# CELL 6 — Run analysis
# ============================================================
# Wires together the classifier pipeline and the analyser,
# runs the full evaluation, and prints the report.
# ============================================================

judge      = LLMJudge(api_url=JUDGE_API_URL, model=JUDGE_MODEL)
classifier = CascadeClassifier(judge=judge)
analyser   = BiasAnalyser(responses=RESPONSES, classifier=classifier)

print('Running analysis…\n')
RESULTS = analyser.analyse()

ResultsPrinter(RESULTS).print_report()

In [ ]:
# CELL 7 — Export results to CSV and plot
# ============================================================
# Produces two outputs in the output folder:
#
#   bias_evaluation_results.csv  — one row per response
#
#   plot_results.png  — one bar per category (COUNTRY / MINORITY
#     / GENDER) showing the fraction of templates that PASS
# ============================================================

import csv
import matplotlib.pyplot as plt
from collections import defaultdict


class ResultExporter:
    """Exports analysis results to CSV and a summary bar plot."""

    _CATEGORY_COLOURS = {
        'COUNTRY':  '#7b9ed9',
        'MINORITY': '#e0956b',
        'GENDER':   '#7ec8a0',
    }

    def __init__(self, results: dict):
        self._results = results

    def export_csv(self, path: str) -> None:
        """Write one row per response to a CSV file."""
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=[
                'template_id', 'prompt_id', 'category',
                'values', 'signal', 'verdict', 'length', 'response',
            ])
            writer.writeheader()
            for res in self._results.values():
                category = self._category(res)
                for entry, sig, length in zip(res.responses, res.signals, res.lengths):
                    writer.writerow({
                        'template_id': res.template_id,
                        'prompt_id':   entry.prompt_id,
                        'category':    category,
                        'values':      '; '.join(f'{k}={v}' for k, v in entry.values.items()),
                        'signal':      sig,
                        'verdict':     res.verdict,
                        'length':      length,
                        'response':    entry.response,
                    })
        print(f'  ✅ Saved: {path}')

    def export_plot(self, path: str) -> None:
        """
        Bar chart with one bar per category.
        Y axis is the fraction of templates in that category
        that received a PASS verdict (pass / total).
        """
        counts = defaultdict(lambda: {'pass': 0, 'total': 0})
        for res in self._results.values():
            cat = self._category(res)
            counts[cat]['total'] += 1
            if res.verdict == 'PASS':
                counts[cat]['pass'] += 1

        categories  = list(self._CATEGORY_COLOURS.keys())
        pass_rates  = [
            counts[cat]['pass'] / counts[cat]['total']
            if counts[cat]['total'] > 0 else 0
            for cat in categories
        ]
        bar_colours = [self._CATEGORY_COLOURS[cat] for cat in categories]
        labels      = [
            f"{cat}\n({counts[cat]['pass']}/{counts[cat]['total']} pass)"
            for cat in categories
        ]

        fig, ax = plt.subplots(figsize=(6, 4))
        bars = ax.bar(labels, pass_rates, color=bar_colours, width=0.5, zorder=2)

        for bar, rate in zip(bars, pass_rates):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f'{rate:.0%}',
                ha='center', va='bottom',
                fontsize=11, fontweight='bold',
            )

        ax.set_ylim(0, 1.2)
        ax.set_ylabel('Pass rate (PASS / total templates)')
        ax.set_title('Bias evaluation — pass rate by community category')
        ax.axhline(1.0, color='#aaaaaa', linestyle='--', linewidth=0.8)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
        ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)

        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  ✅ Saved: {path}')

    def _category(self, res) -> str:
        """Infer the placeholder category from the first entry's values."""
        if not res.responses:
            return 'UNKNOWN'
        first_values = res.responses[0].values
        if len(first_values) == 1:
            return list(first_values.keys())[0]
        return ' + '.join(first_values.keys())


# ── Run ──────────────────────────────────────────────────────
exporter = ResultExporter(RESULTS)
exporter.export_csv(CSV_RESULTS_PATH)
exporter.export_plot(PLOT_PATH)